# 1. Imports

In [24]:
import sqlite3
import pandas as pd
import numpy as np
import re
import json
import warnings
from sklearn.ensemble import IsolationForest
from sklearn.preprocessing import StandardScaler
from datetime import datetime, timedelta
from collections import Counter, defaultdict
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer
from sklearn.decomposition import LatentDirichletAllocation
from sklearn.metrics.pairwise import cosine_similarity
warnings.filterwarnings('ignore')

# 2. Connecting to database and loading all text data

In [2]:
DB_PATH = r"C:\Users\asule\Desktop\Task_DS\drilling_reports.db"
conn = sqlite3.connect(DB_PATH)

query = """
    SELECT
        r.source_file, r.wellbore, r.period,
        o.start_time, o.end_time, o.end_depth_mmd,
        o.main_sub_activity, o.remark
    FROM operations o
    JOIN reports r ON o.report_id = r.id
    WHERE o.remark IS NOT NULL
"""
ops_df = pd.read_sql_query(query, conn)

query_sum = """
    SELECT source_file, wellbore, period, summary_24h, summary_planned
    FROM reports
    WHERE summary_24h IS NOT NULL OR summary_planned IS NOT NULL
"""
reports_df = pd.read_sql_query(query_sum, conn)

query_eq = """
    SELECT e.*, r.wellbore, r.period
    FROM equipment_failure e
    JOIN reports r ON e.report_id = r.id
    WHERE e.remark IS NOT NULL
"""
equip_df = pd.read_sql_query(query_eq, conn)

print(f"Operations:         {len(ops_df)}")
print(f"Reports:            {len(reports_df)}")
print(f"Equipment failure:  {len(equip_df)}")

Operations:         10528
Reports:            1000
Equipment failure:  244


# 3a. Named Entity Recognition (NER)

## Discovering activity labels from data

In [3]:
# DISCOVERY STEP 1: What activity labels exist in our data?

print("=== RAW ACTIVITY LABELS IN DATA ===")
raw_activities = ops_df['main_sub_activity'].dropna()
activity_counts = Counter(raw_activities)
print(f"Total unique raw labels: {len(activity_counts)}")
print("\nAll labels and their counts:")
for act, count in activity_counts.most_common():
    print(f"  {count:>5}  {act}")

=== RAW ACTIVITY LABELS IN DATA ===
Total unique raw labels: 177

All labels and their counts:
   1226  drilling -- casing
   1004  drilling -- drill
    790  drilling -- trip
    591  plug abandon -- other
    521  interruption -- other
    418  drilling -- bop/wellhead equipment
    316  workover -- wire line
    301  interruption -- repair
    252  drilling -- other
    224  plug abandon -- trip
    175  drilling --casing
    142  interruption -- sidetrack
    140  plug abandon -- cut
    138  drilling -- circulating conditioning
    136  drilling -- bop/wellheadequipment
    133  interruption -- maintain
    133  completion -- bop/wellhead equipment
    129  drilling --drill
    127  completion -- circulating conditioning
    125  interruption -- fish
    117  drilling -- bop activities
    114  plug abandon-- other
    109  workover -- rig up/down
    108  completion -- perforate
    103  formation evaluation -- rig up/down
    100  interruption -- waiting on weather
     98  form

## Discovering equipment abbrevations

In [4]:
# DISCOVERY STEP 2: What equipment names appear in our PDFs?

all_remarks = ' '.join(ops_df['remark'].dropna().tolist())

# Finding uppercase abbreviations (2-6 chars) — drilling equipment pattern
abbrevs = re.findall(r'\b[A-Z]{2,6}\b', all_remarks)
abbrev_counts = Counter(abbrevs)

print("=== AUTO-DISCOVERED ABBREVIATIONS FROM ALL PDFs ===")
print(f"Total unique abbreviations found: {len(abbrev_counts)}")
print("\nTop 50 most frequent:")
for abbrev, count in abbrev_counts.most_common(50):
    print(f"  {abbrev:<10} = {count} mentions")

=== AUTO-DISCOVERED ABBREVIATIONS FROM ALL PDFs ===
Total unique abbreviations found: 1499

Top 50 most frequent:
  TO         = 2172 mentions
  RIH        = 1477 mentions
  MD         = 1317 mentions
  POOH       = 1281 mentions
  AND        = 1274 mentions
  BHA        = 1055 mentions
  MT         = 880 mentions
  FROM       = 809 mentions
  DP         = 805 mentions
  BOP        = 720 mentions
  WITH       = 713 mentions
  ON         = 515 mentions
  WOB        = 511 mentions
  RPM        = 469 mentions
  TDS        = 466 mentions
  SPP        = 464 mentions
  AT         = 458 mentions
  UP         = 446 mentions
  ROP        = 438 mentions
  IN         = 433 mentions
  SG         = 385 mentions
  HOLE       = 372 mentions
  BAR        = 368 mentions
  MUD        = 362 mentions
  ECD        = 357 mentions
  OF         = 337 mentions
  TESTED     = 288 mentions
  MWD        = 287 mentions
  NO         = 280 mentions
  TOOL       = 268 mentions
  PRS        = 266 mentions
  FOR       

## Discovering measurement patterns

In [5]:
# DISCOVERY STEP 3: What measurement units appear in our data?

# Finding all unit patterns
unit_pattern = r'(\d+(?:\.\d+)?)\s*([a-zA-Z/]+)\b'
all_units = re.findall(unit_pattern, all_remarks)
unit_counts = Counter([u.lower() for _, u in all_units if len(u) <= 5])

print("=== MEASUREMENT UNITS FOUND IN DATA ===")
print("\nTop 30 most frequent units:")
for unit, count in unit_counts.most_common(30):
    print(f"  {unit:<10} = {count} mentions")

=== MEASUREMENT UNITS FOUND IN DATA ===

Top 30 most frequent units:
  m          = 7021 mentions
  /          = 3168 mentions
  bar        = 2723 mentions
  lpm        = 1525 mentions
  mt         = 949 mentions
  rpm        = 882 mentions
  knm        = 836 mentions
  sg         = 778 mentions
  min        = 627 mentions
  m/hr       = 363 mentions
  l/min      = 325 mentions
  to         = 293 mentions
  bars       = 236 mentions
  ton        = 199 mentions
  and        = 179 mentions
  in         = 166 mentions
  bar/       = 152 mentions
  x          = 143 mentions
  hrs        = 140 mentions
  m/hrs      = 116 mentions
  t          = 108 mentions
  mins       = 107 mentions
  slips      = 95 mentions
  from       = 92 mentions
  m/min      = 87 mentions
  ltrs       = 86 mentions
  hi         = 84 mentions
  tons       = 82 mentions
  deg        = 73 mentions
  mmd        = 72 mentions


## discovering problem keywords

In [6]:
# DISCOVERY STEP 4: What problem-related words appear in our data?

# Finding most common words in remarks that indicate problems
problem_indicators = re.findall(
    r'\b(stuck|tight|failure|breakdown|lost|kick|blowout|drag|torque|'
    r'problem|issue|delay|wait|restrict|difficult|unable|failed|damage)\w*\b',
    all_remarks.lower()
)
problem_counts = Counter(problem_indicators)

print("=== PROBLEM KEYWORDS FOUND IN DATA ===")
for word, count in problem_counts.most_common(30):
    print(f"  {word:<20} = {count} mentions")

=== PROBLEM KEYWORDS FOUND IN DATA ===
  torque               = 509 mentions
  problem              = 220 mentions
  wait                 = 210 mentions
  lost                 = 111 mentions
  tight                = 101 mentions
  unable               = 93 mentions
  drag                 = 85 mentions
  restrict             = 73 mentions
  damage               = 70 mentions
  failure              = 40 mentions
  stuck                = 31 mentions
  kick                 = 30 mentions
  failed               = 16 mentions
  issue                = 13 mentions
  difficult            = 3 mentions
  delay                = 1 mentions


## Building dictionary from discovered data

In [7]:
EQUIPMENT_DICT = {

    'BOP':          ['BOP', 'blowout preventer'],
    'TDS':          ['TDS', 'top drive'],
    'BHA':          ['BHA', 'bottom hole assembly'],
    'XO':           ['XO', 'crossover'],
    'FLX packer':   ['FLX packer', 'FLX-packer'],
    'spear BHA':    ['spear BHA', 'spear'],


    'MWD':          ['MWD', 'measurement while drilling'],
    'LWD':          ['LWD', 'logging while drilling'],
    'IBOP':         ['IBOP'],
    'FOSV':         ['FOSV'],
    'HWDP':         ['HWDP', 'heavy weight drill pipe'],
    'PADPRT':       ['PADPRT'],
    'CART':         ['CART'],
    'ROV':          ['ROV', 'remote operated vehicle'],
    'RSS':          ['RSS', 'rotary steerable', 'powerdrive'],
    'WBRT':         ['WBRT'],
    'casing hanger':['casing hanger', 'csg hanger'],
    'wellhead':     ['wellhead', 'well head'],
    'mud pump':     ['mud pump', 'rig pump'],
}


ACTIVITY_MAP = {
    'TRIP_IN':           ['trip in', 'rih', 'run in hole', 'running in'],
    'TRIP_OUT':          ['trip out', 'pooh', 'pull out', 'pulling out'],
    'DRILL':             ['drill', 'drilling', 'rotate'],
    'CEMENT':            ['cement', 'cementing', 'cmt'],
    'PRESSURE_TEST':     ['pressure test', 'leak off', 'fit test', 'bop test'],
    'CIRCULATE':         ['circulat', 'bottoms up', 'sweep'],
    'REAM':              ['ream', 'reaming', 'back ream'],
    'CASING':            ['casing', 'csg', 'liner'],
    'EQUIPMENT_FAILURE': ['failure', 'breakdown', 'repair'],
    'WAIT':              ['wait', 'standby', 'hold'],
    'SURVEY':            ['survey', 'mwd', 'gyro'],
    'FISHING':           ['fish', 'overshot', 'spear'],
    'WASH':              ['wash', 'washing'],
    'DISPLACEMENT':      ['displac', 'flush'],
    'BOP_TEST':          ['bop test', 'bop activ'],
    'LOGGING':           ['log', 'lwd', 'wireline'],
}


SEVERITY_MAP = {
    'critical': ['stuck', 'blowout', 'kick', 'emergency', 'shut in', 'well control'],
    'high':     ['tight hole', 'lost circulation', 'differential stuck',
                 'pack off', 'washout', 'twist off', 'failure', 'breakdown'],
    'medium':   ['drag', 'torque', 'fill', 'restriction', 'problem', 'delay', 'unable'],
    'low':      ['minor', 'slight', 'temporary'],
}

print("Dictionaries built from data discovery!")
print(f"Equipment types:  {len(EQUIPMENT_DICT)}")
print(f"Activity labels:  {len(ACTIVITY_MAP)}")
print(f"Severity levels:  {len(SEVERITY_MAP)}")

Dictionaries built from data discovery!
Equipment types:  19
Activity labels:  16
Severity levels:  4


## NER functions

In [8]:
def extract_depths(text):
    """Extract depth values — unit 'm', 'mMD', 'mTVD' confirmed in data"""
    if not text: return []
    pattern = r'(\d+(?:[.,]\d+)?)\s*(?:m(?:MD|TVD|D)?)\b'
    matches = re.findall(pattern, text, re.IGNORECASE)
    return [float(m.replace(',', '.')) for m in matches]

def extract_measurements(text):
    """Extract measurements — units confirmed from data scan in Cell 5"""
    if not text: return {}
    measurements = {}
    rpm = re.findall(r'(\d+(?:\.\d+)?)\s*(?:RPM|rpm)', text)
    if rpm: measurements['RPM'] = [float(x) for x in rpm]
    bar = re.findall(r'(\d+(?:\.\d+)?)\s*(?:bar|BAR)', text)
    if bar: measurements['pressure_bar'] = [float(x) for x in bar]
    lpm = re.findall(r'(\d+(?:\.\d+)?)\s*(?:lpm|LPM)', text)
    if lpm: measurements['flow_lpm'] = [float(x) for x in lpm]
    mt = re.findall(r'(\d+(?:\.\d+)?)\s*(?:MT|ton)\b', text, re.IGNORECASE)
    if mt: measurements['weight_mt'] = [float(x) for x in mt]
    rop = re.findall(r'(\d+(?:\.\d+)?)\s*(?:m/hr|m/h)', text, re.IGNORECASE)
    if rop: measurements['rop_mhr'] = [float(x) for x in rop]
    return measurements

def extract_time_references(text):
    """Extract time references — HH:MM format found in data"""
    if not text: return []
    return re.findall(r'\b(\d{2}:?\d{2})\s*(?:HRS|hrs)?\b', text)

def extract_equipment(text):
    """Extract equipment using dictionary built from data discovery"""
    if not text: return []
    found = []
    text_lower = text.lower()
    for equip_name, aliases in EQUIPMENT_DICT.items():
        for alias in aliases:
            if alias.lower() in text_lower:
                found.append(equip_name)
                break
    return list(set(found))

# Test on sample
sample = ops_df['remark'].iloc[63]
print("Sample remark:")
print(sample)
print(f"\nDepths:       {extract_depths(sample)}")
print(f"Measurements: {extract_measurements(sample)}")
print(f"Times:        {extract_time_references(sample)}")
print(f"Equipment:    {extract_equipment(sample)}")

Sample remark:
FINALLY ESTABLISHED CIRCULATION WITH FULL RETURNS AT 1860 LPM/164 BAR. STARTED MOVING UPWARDS TOWARD WINDOW-STOPPED AT 2188 M. ORIENTED TOOL- FACE & THEN FLOW CHECKED. STARTED PULLING UP WHILE PUMPING AT 450 LPM- LOSING CIRCULATION AGAIN. STOPPED PUMPING & CONTINUED POOH-PULLED THROUGH WINDOW WITH NO ROTATING/PUMPING WITH MAXIMUM DRAG = 5 MT.

Depths:       [2188.0]
Measurements: {'pressure_bar': [164.0], 'flow_lpm': [1860.0, 450.0], 'weight_mt': [5.0]}
Times:        ['1860', '2188']
Equipment:    []


## Applying NER to all

In [9]:
print("Applying NER to all operations...")

ops_df['ner_depths']       = ops_df['remark'].apply(extract_depths)
ops_df['ner_measurements'] = ops_df['remark'].apply(extract_measurements)
ops_df['ner_times']        = ops_df['remark'].apply(extract_time_references)
ops_df['ner_equipment']    = ops_df['remark'].apply(extract_equipment)

print(f"NER applied to {len(ops_df)} operations\n")

# Statistics
all_equip = [e for equips in ops_df['ner_equipment'] for e in equips]
print("=== NER STATISTICS ===")
print(f"Operations with depth mentions:    {ops_df['ner_depths'].apply(len).gt(0).sum()}")
print(f"Operations with measurements:      {ops_df['ner_measurements'].apply(len).gt(0).sum()}")
print(f"Operations with equipment:         {ops_df['ner_equipment'].apply(len).gt(0).sum()}")
print(f"Total equipment mentions:          {len(all_equip)}")

print("\nTop equipment mentioned:")
for equip, count in Counter(all_equip).most_common(10):
    print(f"  {equip:<25} = {count}")

Applying NER to all operations...
NER applied to 10528 operations

=== NER STATISTICS ===
Operations with depth mentions:    3734
Operations with measurements:      2956
Operations with equipment:         3070
Total equipment mentions:          3677

Top equipment mentioned:
  BHA                       = 1041
  BOP                       = 638
  TDS                       = 545
  MWD                       = 287
  wellhead                  = 241
  ROV                       = 200
  HWDP                      = 195
  spear BHA                 = 108
  RSS                       = 98
  mud pump                  = 84


## NER results

In [10]:
print("=== NAMED ENTITIES CLASSIFIED BY TYPE ===\n")

sample_ops = ops_df[
    (ops_df['ner_equipment'].apply(len) > 0) &
    (ops_df['ner_depths'].apply(len) > 0) &
    (ops_df['ner_measurements'].apply(len) > 0)
].head(5)

for _, row in sample_ops.iterrows():
    print(f"File: {row['source_file']}")
    print(f"Remark: {row['remark'][:200]}")
    print(f"  EQUIPMENT:    {row['ner_equipment']}")
    print(f"  DEPTHS:       {row['ner_depths']}")
    print(f"  MEASUREMENTS: {row['ner_measurements']}")
    print(f"  TIMES:        {row['ner_times']}")
    print()

=== NAMED ENTITIES CLASSIFIED BY TYPE ===

File: 15_9_19_A_1997_07_27.pdf
Remark: ORIENTED BIT/MOTOR & SLACKED-OFF THROUGH WINDOW.REAMED F/2208-2212M. STARTED DRILLING-SPP INCREASED 35BAR&TDS STALLED Ø16300NM.FREED STUCKSTRING USING 32 MT O/PULL-TORQUE STILL TRAPPED IN STRING.WHILE
  EQUIPMENT:    ['TDS']
  DEPTHS:       [2212.0, 2207.0, 2207.0, 2202.0]
  MEASUREMENTS: {'pressure_bar': [35.0], 'weight_mt': [32.0, 55.0]}
  TIMES:        ['2208']

File: 15_9_19_A_1997_10_08.pdf
Remark: INSTALLED XO SUB AND 5" PUP JOINT. CONNECTED TOP DRIVE AND WORKED CSG SCRAPER OVER PACKER SETTING AREA FROM 3795 M TO3820 M. PUMPED 850 LPM - 310 BAR SPP.
  EQUIPMENT:    ['TDS', 'XO']
  DEPTHS:       [3795.0, 3820.0]
  MEASUREMENTS: {'pressure_bar': [310.0], 'flow_lpm': [850.0]}
  TIMES:        ['3795']

File: 15_9_19_A_1997_10_23.pdf
Remark: POOH WITH DST STRING. L/D LUBRICATOR VALVES AND SUBSEA TEST TREE. CHECKED ROTARY TORQUE - MAX 9000 FT X LBS. CONTINUED POOH. L/D DST TOOLS. MEANWHILE, AFTER DST TOOL

# 3b. Activity Classification

In [11]:
# ACTIVITY_MAP built from "Discovering activity" cell
def classify_activity(main_activity, remark):
    """
    Rule-based classifier built from raw label discovery (Cell 3).
    Checks main_sub_activity first (more reliable), then remark text.
    """
    text = (str(main_activity) + ' ' + str(remark)).lower()
    for label, keywords in ACTIVITY_MAP.items():
        for kw in keywords:
            if kw in text:
                return label
    return 'OTHER'

ops_df['activity_label'] = ops_df.apply(
    lambda row: classify_activity(row['main_sub_activity'], row['remark']), axis=1
)

print("=== ACTIVITY CLASSIFICATION RESULTS ===")
label_counts = ops_df['activity_label'].value_counts()
for label, count in label_counts.items():
    pct = count / len(ops_df) * 100
    bar = '-' * int(pct / 2)
    print(f"  {label:<20} {count:>5} ({pct:>5.1f}%) {bar}")

=== ACTIVITY CLASSIFICATION RESULTS ===
  DRILL                 4158 ( 39.5%) -------------------
  OTHER                 1966 ( 18.7%) ---------
  TRIP_IN               1405 ( 13.3%) ------
  TRIP_OUT              1094 ( 10.4%) -----
  CIRCULATE              374 (  3.6%) -
  CEMENT                 328 (  3.1%) -
  EQUIPMENT_FAILURE      288 (  2.7%) -
  WAIT                   218 (  2.1%) -
  LOGGING                167 (  1.6%) 
  PRESSURE_TEST          146 (  1.4%) 
  CASING                 139 (  1.3%) 
  FISHING                 68 (  0.6%) 
  REAM                    54 (  0.5%) 
  DISPLACEMENT            53 (  0.5%) 
  SURVEY                  38 (  0.4%) 
  WASH                    32 (  0.3%) 


# 3c. TF-IDF Keyword Extraction

In [12]:
report_texts = ops_df.groupby('source_file')['remark'].apply(
    lambda x: ' '.join(x.dropna())
).reset_index()
report_texts.columns = ['source_file', 'combined_text']
report_texts = report_texts.merge(
    ops_df[['source_file', 'wellbore']].drop_duplicates(), on='source_file'
)

vectorizer = TfidfVectorizer(
    max_features=5000,
    ngram_range=(1, 2),
    min_df=2,
    max_df=0.85,
    stop_words='english'
)
tfidf_matrix = vectorizer.fit_transform(report_texts['combined_text'])
feature_names = vectorizer.get_feature_names_out()

print(f"TF-IDF corpus: {len(report_texts)} reports, {len(feature_names)} features")

def get_top_keywords(source_file, top_n=10):
    idx = report_texts[report_texts['source_file'] == source_file].index
    if len(idx) == 0: return []
    row = tfidf_matrix[idx[0]].toarray().flatten()
    top_indices = row.argsort()[-top_n:][::-1]
    return [(feature_names[i], round(row[i], 4)) for i in top_indices if row[i] > 0]

print("\n=== TOP KEYWORDS PER REPORT (sample 5) ===")
for f in report_texts['source_file'].head(5).tolist():
    keywords = get_top_keywords(f, top_n=8)
    print(f"\n{f}:")
    for kw, score in keywords:
        print(f"  {kw:<30} = {score}")

TF-IDF corpus: 964 reports, 5000 features

=== TOP KEYWORDS PER REPORT (sample 5) ===

15_9_19_A_1997_07_26.pdf:
  whipstock                      = 0.5092
  window                         = 0.4034
  m3 hi                          = 0.2066
  vis pill                       = 0.1962
  milling                        = 0.1752
  hi vis                         = 0.1734
  vis                            = 0.1721
  hi                             = 0.1673

15_9_19_A_1997_07_27.pdf:
  pod                            = 0.3209
  yellow pod                     = 0.2911
  yellow                         = 0.2911
  bit                            = 0.2038
  window                         = 0.1832
  using                          = 0.1663
  drilling bha                   = 0.1645
  checked pumped                 = 0.1617

15_9_19_A_1997_07_28.pdf:
  milling                        = 0.3959
  milling bha                    = 0.2838
  bridge plug                    = 0.2729
  flowchecked pumped             = 

# ADDITIONAL operations

## Operation Duration Analysis for potential problems

In [22]:
from datetime import datetime, timedelta

def parse_time(t):
    """Parse HH:MM time string to minutes since midnight."""
    if not t:
        return None
    try:
        h, m = map(int, str(t).split(':'))
        return h * 60 + m
    except:
        return None

def calc_duration(start, end):
    """
    Calculate operation duration in hours.
    Handles overnight operations (end < start means crossed midnight).
    """
    s = parse_time(start)
    e = parse_time(end)
    if s is None or e is None:
        return None
    if e < s:  # crossed midnight
        e += 24 * 60
    return round((e - s) / 60, 2)

ops_df['duration_hrs'] = ops_df.apply(
    lambda row: calc_duration(row['start_time'], row['end_time']), axis=1
)

print("=== OPERATION DURATION ANALYSIS ===\n")

# Duration stats per activity label
duration_stats = ops_df.groupby('activity_label')['duration_hrs'].agg([
    'count', 'mean', 'median', 'max'
]).round(2).sort_values('mean', ascending=False)

print("Average duration per activity type:")
print(f"  {'Activity':<22} {'Count':>6} {'Mean(h)':>8} {'Median(h)':>10} {'Max(h)':>7}")
print(f"  {'-'*55}")
for label, row in duration_stats.iterrows():
    print(f"  {label:<22} {int(row['count']):>6} {row['mean']:>8.1f} {row['median']:>10.1f} {row['max']:>7.1f}")

# Find suspiciously long operations
print("\n=== UNUSUALLY LONG OPERATIONS (>12 hours) ===")
long_ops = ops_df[ops_df['duration_hrs'] > 12].sort_values('duration_hrs', ascending=False)
print(f"Total: {len(long_ops)} operations\n")
for _, row in long_ops.head(3).iterrows():
    print(f"  Duration: {row['duration_hrs']}h | Activity: {row['activity_label']}")
    print(f"  File: {row['source_file']}")
    print(f"  Remark: {str(row['remark'])[:150]}")
    print()

=== OPERATION DURATION ANALYSIS ===

Average duration per activity type:
  Activity                Count  Mean(h)  Median(h)  Max(h)
  -------------------------------------------------------
  WAIT                      218      4.1        2.5    18.0
  REAM                       54      3.0        2.0    13.5
  LOGGING                   167      2.3        1.5    18.0
  TRIP_IN                  1405      2.3        1.5    18.0
  TRIP_OUT                 1094      2.2        1.5    18.0
  EQUIPMENT_FAILURE         288      2.2        1.0    15.0
  SURVEY                     38      2.2        1.5    14.8
  DRILL                    4158      2.2        1.2    20.0
  CIRCULATE                 374      1.8        1.2    16.0
  PRESSURE_TEST             146      1.7        1.0     9.0
  OTHER                    1966      1.7        0.8    18.0
  FISHING                    68      1.6        0.5    16.0
  CASING                    139      1.4        1.0     9.0
  CEMENT                    3

## Non-Productive Time (NPT) Detection

In [23]:
NPT_KEYWORDS = {
    'stuck_pipe':        ['stuck', 'differential stuck', 'mechanically stuck'],
    'lost_circulation':  ['lost circulation', 'loss of circulation', 'total loss'],
    'equipment_failure': ['failure', 'breakdown', 'malfunction', 'repair', 'replace'],
    'waiting':           ['wait', 'standby', 'hold', 'weather', 'supply boat'],
    'well_control':      ['kick', 'blowout', 'shut in', 'well control'],
    'tight_hole':        ['tight hole', 'overpull', 'high drag', 'high torque'],
    'fishing':           ['fishing', 'overshot', 'junk', 'twist off'],
    'bop_issues':        ['bop failure', 'bop test fail', 'no function'],
}

def detect_npt(remark, activity):
    """
    Detect Non-Productive Time events from operation remarks.
    Returns NPT category or None if productive operation.
    WHY: Rule-based is preferred here — NPT definitions are
    standardized in the drilling industry.
    """
    if not remark:
        return None
    text = str(remark).lower()

    for npt_type, keywords in NPT_KEYWORDS.items():
        for kw in keywords:
            if kw in text:
                return npt_type
    return None

ops_df['npt_type'] = ops_df.apply(
    lambda row: detect_npt(row['remark'], row['main_sub_activity']), axis=1
)

# Calculate NPT hours
ops_df['is_npt'] = ops_df['npt_type'].notna()

print("=== NON-PRODUCTIVE TIME (NPT) ANALYSIS ===\n")

# Overall NPT stats
total_ops = len(ops_df)
npt_ops = ops_df['is_npt'].sum()
print(f"Total operations:     {total_ops}")
print(f"NPT operations:       {npt_ops} ({npt_ops/total_ops*100:.1f}%)")
print(f"Productive:           {total_ops-npt_ops} ({(total_ops-npt_ops)/total_ops*100:.1f}%)")

# NPT hours
npt_hours = ops_df[ops_df['is_npt']]['duration_hrs'].sum()
total_hours = ops_df['duration_hrs'].sum()
print(f"\nTotal drilling hours: {total_hours:.0f}h")
print(f"NPT hours:            {npt_hours:.0f}h ({npt_hours/total_hours*100:.1f}%)")

# NPT by type
print("\nNPT breakdown by type:")
npt_by_type = ops_df[ops_df['is_npt']].groupby('npt_type').agg(
    count=('is_npt', 'sum'),
    hours=('duration_hrs', 'sum')
).round(1).sort_values('hours', ascending=False)

for npt_type, row in npt_by_type.iterrows():
    print(f"  {npt_type:<25} = {int(row['count']):>4} events | {row['hours']:>7.1f}h")

# NPT by wellbore
print("\nNPT hours per wellbore:")
npt_by_well = ops_df[ops_df['is_npt']].groupby('wellbore')['duration_hrs'].sum().sort_values(ascending=False)
for well, hours in npt_by_well.head(8).items():
    print(f"  {well:<20} = {hours:.1f}h NPT")

# Sample NPT events
print("\n=== SAMPLE NPT EVENTS ===")
for _, row in ops_df[ops_df['is_npt']].head(5).iterrows():
    print(f"\n  [{row['npt_type'].upper()}] {row['source_file']}")
    print(f"  Duration: {row['duration_hrs']}h")
    print(f"  Remark: {str(row['remark'])[:200]}")

=== NON-PRODUCTIVE TIME (NPT) ANALYSIS ===

Total operations:     10528
NPT operations:       935 (8.9%)
Productive:           9593 (91.1%)

Total drilling hours: 21831h
NPT hours:            2674h (12.2%)

NPT breakdown by type:
  waiting                   =  291 events |  1031.4h
  equipment_failure         =  269 events |   803.2h
  tight_hole                =  204 events |   446.8h
  fishing                   =   54 events |   157.5h
  well_control              =   74 events |   139.8h
  stuck_pipe                =   31 events |    62.8h
  lost_circulation          =   11 events |    32.2h
  bop_issues                =    1 events |     0.5h

NPT hours per wellbore:
  15/9-F-12            = 639.2h NPT
  15/9-19 ST2          = 377.5h NPT
  15/9-F-14            = 297.8h NPT
  15/9-F-11 T2         = 180.8h NPT
  15/9-F-11 B          = 176.4h NPT
  15/9-F-10            = 168.0h NPT
  15/9-19 BT2          = 161.5h NPT
  15/9-F-15            = 129.8h NPT

=== SAMPLE NPT EVENTS ===

  [EQ

## Anomaly Detection

In [25]:
print("=== ANOMALY DETECTION IN DRILLING OPERATIONS ===\n")

# Extract numerical features for anomaly detection
def extract_features(row):
    """Extract numerical features from each operation for ML."""
    features = {}

    # Duration
    features['duration_hrs'] = row['duration_hrs'] or 0

    # First depth mention
    depths = row['ner_depths']
    features['depth'] = depths[0] if depths else 0

    # Measurements
    meas = row['ner_measurements']
    features['rpm']      = meas.get('RPM', [0])[0] if meas.get('RPM') else 0
    features['pressure'] = meas.get('pressure_bar', [0])[0] if meas.get('pressure_bar') else 0
    features['flow']     = meas.get('flow_lpm', [0])[0] if meas.get('flow_lpm') else 0
    features['weight']   = meas.get('weight_mt', [0])[0] if meas.get('weight_mt') else 0

    # Equipment count
    features['equipment_count'] = len(row['ner_equipment'])

    return features

# Build feature matrix
feature_records = ops_df.apply(extract_features, axis=1).tolist()
features_df = pd.DataFrame(feature_records).fillna(0)

# Only use rows with some non-zero data
mask = features_df.sum(axis=1) > 0
features_filtered = features_df[mask]
ops_filtered = ops_df[mask].copy()

# Scale features
scaler = StandardScaler()
X_scaled = scaler.fit_transform(features_filtered)

# Isolation Forest — detects anomalies without labels
# contamination=0.05 means we expect ~5% anomalies
iso_forest = IsolationForest(contamination=0.05, random_state=42, n_estimators=100)
anomaly_labels = iso_forest.fit_predict(X_scaled)
anomaly_scores = iso_forest.score_samples(X_scaled)

ops_filtered = ops_filtered.copy()
ops_filtered['is_anomaly'] = anomaly_labels == -1
ops_filtered['anomaly_score'] = anomaly_scores

n_anomalies = ops_filtered['is_anomaly'].sum()
print(f"Total operations analyzed: {len(ops_filtered)}")
print(f"Anomalies detected:        {n_anomalies} ({n_anomalies/len(ops_filtered)*100:.1f}%)")

# Show most anomalous operations
print("\n=== MOST ANOMALOUS OPERATIONS ===")
most_anomalous = ops_filtered[ops_filtered['is_anomaly']].nsmallest(5, 'anomaly_score')
for _, row in most_anomalous.iterrows():
    print(f"\n  Score: {row['anomaly_score']:.3f} | File: {row['source_file']}")
    print(f"  Activity: {row['activity_label']} | Duration: {row['duration_hrs']}h")
    print(f"  Remark: {str(row['remark'])[:200]}")

# Anomaly distribution by activity
print("\n=== ANOMALY RATE BY ACTIVITY TYPE ===")
anomaly_by_activity = ops_filtered.groupby('activity_label').agg(
    total=('is_anomaly', 'count'),
    anomalies=('is_anomaly', 'sum')
)
anomaly_by_activity['rate'] = (
    anomaly_by_activity['anomalies'] / anomaly_by_activity['total'] * 100
).round(1)
anomaly_by_activity = anomaly_by_activity.sort_values('rate', ascending=False)

for activity, row in anomaly_by_activity.head(8).iterrows():
    print(f"  {activity:<22} {int(row['anomalies']):>4}/{int(row['total']):<6} ({row['rate']}%)")

=== ANOMALY DETECTION IN DRILLING OPERATIONS ===

Total operations analyzed: 10528
Anomalies detected:        527 (5.0%)

=== MOST ANOMALOUS OPERATIONS ===

  Score: -0.756 | File: 15_9_F_10_2009_05_26.pdf
  Activity: DRILL | Duration: 16.25h
  Remark: Drilled 8 1/2" hole section from 3863 m to 4159 m MD. Drilling parameters : Flow 2200-2500 lpm / SPP 190-208 bar / 200 RPM / WOB 9-13 MT / Torque 20-25 kNm / ROP 18-28 m/hrs. Performed MWD survey on c

  Score: -0.737 | File: 15_9_F_10_2009_05_29.pdf
  Activity: DRILL | Duration: 2.0h
  Remark: Drilled 8 1/2" hole section from 4750 m to 4790 m MD. Drilling parameters : Flow 2800 lpm / SPP ~266 bar / 200 RPM / WOB ~10 MT / Torque 20-23 kNm / ROP 20-25 m/hr. Performed MWD survey on connections

  Score: -0.735 | File: 15_9_F_10_2009_05_27.pdf
  Activity: DRILL | Duration: 10.0h
  Remark: Drilled 8 1/2" hole section from 4306 m to 4482 m MD. Drilling parameters : Flow 2800 lpm / SPP ~240 bar / 200 RPM / WOB 9-12 MT / Torque 21-23 kNm / ROP 

## Severity Detection - flags problematic operations

In [26]:
def assess_severity(text):
    if not text: return 'normal'
    text_lower = text.lower()
    for level in ['critical', 'high', 'medium', 'low']:
        for kw in SEVERITY_MAP[level]:
            if kw in text_lower:
                return level
    return 'normal'

ops_df['severity'] = ops_df['remark'].apply(assess_severity)

print("=== SEVERITY DISTRIBUTION ===")
for sev, count in ops_df['severity'].value_counts().items():
    pct = count / len(ops_df) * 100
    bar = '-' * int(pct / 2)
    print(f"  {sev:<10} {count:>5} ({pct:>5.1f}%) {bar}")

print("\n=== CRITICAL/HIGH SEVERITY SAMPLES ===")
for _, row in ops_df[ops_df['severity'].isin(['critical', 'high'])].head(5).iterrows():
    print(f"\n  [{row['severity'].upper()}] {row['source_file']}")
    print(f"  Remark: {row['remark'][:200]}")

=== SEVERITY DISTRIBUTION ===
  normal      9151 ( 86.9%) -------------------------------------------
  medium      1168 ( 11.1%) -----
  critical     117 (  1.1%) 
  high          68 (  0.6%) 
  low           24 (  0.2%) 

=== CRITICAL/HIGH SEVERITY SAMPLES ===

  [CRITICAL] 15_9_19_A_1997_07_27.pdf
  Remark: ORIENTED BIT/MOTOR & SLACKED-OFF THROUGH WINDOW.REAMED F/2208-2212M. STARTED DRILLING-SPP INCREASED 35BAR&TDS STALLED Ø16300NM.FREED STUCKSTRING USING 32 MT O/PULL-TORQUE STILL TRAPPED IN STRING.WHILE

  [CRITICAL] 15_9_19_A_1997_08_02.pdf
  Remark: CIRCULATED W/2500LPM & ORIENTED TOOLFACE PRIOR TO PULLING THROUGH CSG. WINDOW. HOLE PACKED-OFF & STRING TORQUED-UP & LOST CIRC-STRING STUCK 2MIN. WORKED STRING FREE BY GOING DOWNWARDS;MAX WEIGHT SLACK

  [HIGH] 15_9_19_A_1997_08_02.pdf
  Remark: WORKED STRING DOWN TO 2199 M BIT DEPTH - 16 M BELOW WINDOW'S BOTTOM. HOLE PACKED-OFF;LOST CIRCULATION SEVERAL TIMES. ATTEMPTED TO ESTABLISH CIRCULATION - IN STEPS - AT DIFFERENT FLOW RATES WHI

# Saving All Results

In [27]:
output_path = r"C:\Users\asule\Desktop\Task_DS\nlp_analysis_results.xlsx"

with pd.ExcelWriter(output_path, engine='openpyxl') as writer:

    # Sheet 1: NER + Classification
    ner_export = ops_df[[
        'source_file', 'wellbore', 'period', 'start_time', 'end_time',
        'main_sub_activity', 'remark', 'ner_depths', 'ner_equipment',
        'ner_measurements', 'ner_times', 'activity_label', 'severity'
    ]].copy()
    for col in ['ner_depths', 'ner_equipment', 'ner_measurements', 'ner_times']:
        ner_export[col] = ner_export[col].apply(str)
    ner_export.to_excel(writer, sheet_name='NER + Classification', index=False)

    # Sheet 2: TF-IDF Keywords
    keywords_data = []
    for _, row in report_texts.iterrows():
        for kw, score in get_top_keywords(row['source_file'], top_n=10):
            keywords_data.append({
                'source_file': row['source_file'],
                'wellbore': row['wellbore'],
                'keyword': kw,
                'tfidf_score': score
            })
    pd.DataFrame(keywords_data).to_excel(writer, sheet_name='TF-IDF Keywords', index=False)

    # Sheet 3: Duration Analysis
    duration_export = ops_df[[
        'source_file', 'wellbore', 'period', 'start_time', 'end_time',
        'duration_hrs', 'activity_label', 'main_sub_activity', 'remark'
    ]].copy()
    duration_export.to_excel(writer, sheet_name='Duration Analysis', index=False)

    # Sheet 4: NPT Analysis
    npt_export = ops_df[ops_df['is_npt']][[
        'source_file', 'wellbore', 'period', 'start_time', 'end_time',
        'duration_hrs', 'npt_type', 'remark'
    ]].copy()
    npt_export.to_excel(writer, sheet_name='NPT Analysis', index=False)

    # Sheet 5: Anomaly Detection
    anomaly_export = ops_filtered[ops_filtered['is_anomaly']][[
        'source_file', 'wellbore', 'period', 'activity_label',
        'duration_hrs', 'anomaly_score', 'remark'
    ]].copy()
    anomaly_export.to_excel(writer, sheet_name='Anomaly Detection', index=False)

    # Sheet 6: Severity Analysis
    ops_df[ops_df['severity'] != 'normal'][[
        'source_file', 'wellbore', 'period', 'remark', 'severity'
    ]].to_excel(writer, sheet_name='Severity Analysis', index=False)

print(f"Saved to: {output_path}")
print("6 sheets:")
print("  1. NER + Classification")
print("  2. TF-IDF Keywords")
print("  3. Duration Analysis")
print("  4. NPT Analysis")
print("  5. Anomaly Detection")
print("  6. Severity Analysis")

Saved to: C:\Users\asule\Desktop\Task_DS\nlp_analysis_results.xlsx
6 sheets:
  1. NER + Classification
  2. TF-IDF Keywords
  3. Duration Analysis
  4. NPT Analysis
  5. Anomaly Detection
  6. Severity Analysis
